<a href="https://colab.research.google.com/github/anokhina-rgb/Google-Colabs/blob/main/%D0%A1%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B0_%D0%B5%D0%BA%D1%81%D1%82%D1%80%D0%B0%D0%BA%D1%86%D1%96%D1%97_%D1%82%D0%B5%D1%80%D0%BC%D1%96%D0%BD%D1%96%D0%B2_(OPUS)_ed_tmx%2Bcvs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""Система екстракції, перекладу OPUS, TMX, Глосарію та Редагування"""

import os
import sqlite3
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import threading
import spacy
import csv
from PyPDF2 import PdfReader
from docx import Document
from transformers import MarianMTModel, MarianTokenizer
import xml.etree.ElementTree as ET

# Шлях до БД
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DB_PATH = os.path.join(BASE_DIR, "nlp_terms_opus.db")

try:
    nlp = spacy.load("en_core_web_sm")
except:
    raise OSError("Встановіть модель: python -m spacy download en_core_web_sm")

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute('''CREATE TABLE IF NOT EXISTS terms
                      (id INTEGER PRIMARY KEY AUTOINCREMENT, en_term TEXT UNIQUE, uk_term TEXT, length INTEGER)''')
    conn.commit()
    conn.close()

class TermManagerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Система: Редактор термінів та Експорт")
        init_db()

        btn_frame = tk.Frame(root, pady=10)
        btn_frame.pack(fill=tk.X)

        tk.Button(btn_frame, text="1. PDF", command=self.analyze_pdf).pack(side=tk.LEFT, padx=5)
        tk.Button(btn_frame, text="2. Переклад (OPUS)", command=self.process_batch).pack(side=tk.LEFT, padx=5)
        tk.Button(btn_frame, text="3. Word", command=self.export_to_word).pack(side=tk.LEFT, padx=5)
        tk.Button(btn_frame, text="4. TMX", command=self.export_to_tmx).pack(side=tk.LEFT, padx=5)
        tk.Button(btn_frame, text="5. CSV", command=self.export_to_glossary_csv).pack(side=tk.LEFT, padx=5)

        self.tree = ttk.Treeview(root, columns=("ID", "EN", "UK", "LEN"), show="headings")
        for col in ["ID", "EN", "UK", "LEN"]: self.tree.heading(col, text=col)
        self.tree.column("ID", width=40)
        self.tree.pack(fill=tk.BOTH, expand=True, padx=5)

        # Підключення події редагування
        self.tree.bind("<Double-1>", self.on_double_click)

        self.log_area = tk.Text(root, height=8)
        self.log_area.pack(fill=tk.X, padx=5, pady=5)

        self.refresh_table()

        self.log("Завантаження моделі OPUS-MT...")
        self.root.update()
        self.model_name = "Helsinki-NLP/opus-mt-en-uk"
        self.tokenizer = MarianTokenizer.from_pretrained(self.model_name)
        self.model = MarianMTModel.from_pretrained(self.model_name)
        self.log("Модель OPUS готова.")

    def on_double_click(self, event):
        """Редагування комірки"""
        region = self.tree.identify("region", event.x, event.y)
        if region != "cell": return
        column = self.tree.identify_column(event.x)
        item_id = self.tree.identify_row(event.y)

        # Дозволяємо редагувати лише колонки EN (2) та UK (3)
        if column in ["#2", "#3"]:
            col_name = "en_term" if column == "#2" else "uk_term"
            current_val = self.tree.item(item_id, "values")[int(column[1])-1]

            # Створюємо Entry для редагування
            x, y, w, h = self.tree.bbox(item_id, column)
            entry = tk.Entry(self.root)
            entry.place(x=x, y=y, width=w, height=h)
            entry.insert(0, current_val)
            entry.focus_set()

            def save_edit(event):
                new_val = entry.get()
                db_id = self.tree.item(item_id, "values")[0]
                conn = sqlite3.connect(DB_PATH)
                conn.execute(f"UPDATE terms SET {col_name} = ? WHERE id = ?", (new_val, db_id))
                conn.commit()
                conn.close()
                self.refresh_table()
                entry.destroy()

            entry.bind("<Return>", save_edit)
            entry.bind("<FocusOut>", lambda e: entry.destroy())

    def log(self, message):
        self.log_area.insert(tk.END, f"[LOG]: {message}\n"); self.log_area.see(tk.END)

    def export_to_glossary_csv(self):
        file_path = filedialog.asksaveasfilename(defaultextension=".csv")
        if not file_path: return
        conn = sqlite3.connect(DB_PATH)
        rows = conn.execute("SELECT en_term, uk_term FROM terms WHERE uk_term != ''").fetchall()
        with open(file_path, mode='w', encoding='utf-8', newline='') as f:
            writer = csv.writer(f, delimiter=';')
            writer.writerow(['English Term', 'Ukrainian Term'])
            writer.writerows(rows)
        messagebox.showinfo("Успіх", "Глосарій збережено!")
        conn.close()

    def export_to_tmx(self):
        file_path = filedialog.asksaveasfilename(defaultextension=".tmx")
        if not file_path: return
        tmx = ET.Element("tmx", version="1.4")
        ET.SubElement(tmx, "header", srclang="en", adminlang="uk", o_tmf="unknown", segtype="sentence")
        body = ET.SubElement(tmx, "body")
        conn = sqlite3.connect(DB_PATH)
        for en, uk in conn.execute("SELECT en_term, uk_term FROM terms WHERE uk_term != ''").fetchall():
            tu = ET.SubElement(body, "tu")
            tuv_en = ET.SubElement(tu, "tuv", {"xml:lang": "en"})
            ET.SubElement(tuv_en, "seg").text = en
            tuv_uk = ET.SubElement(tu, "tuv", {"xml:lang": "uk"})
            ET.SubElement(tuv_uk, "seg").text = uk
        tree = ET.ElementTree(tmx)
        tree.write(file_path, encoding="utf-8", xml_declaration=True)
        messagebox.showinfo("Успіх", "TMX збережено!")
        conn.close()

    def analyze_pdf(self):
        path = filedialog.askopenfilename(filetypes=[("PDF files", "*.pdf")])
        if not path: return
        threading.Thread(target=self._scan_pdf, args=(path,), daemon=True).start()

    def _scan_pdf(self, path):
        reader = PdfReader(path)
        conn = sqlite3.connect(DB_PATH)
        for page in reader.pages:
            text = page.extract_text()
            if not text: continue
            doc = nlp(text)
            for chunk in doc.noun_chunks:
                tokens = [t for t in chunk if t.is_alpha]
                if 2 <= len(tokens) <= 4 and all(t.pos_ in ['NOUN', 'PROPN', 'ADJ'] for t in tokens):
                    term = " ".join([t.text for t in tokens])
                    conn.execute("INSERT OR IGNORE INTO terms (en_term, uk_term, length) VALUES (?, ?, ?)", (term, "", len(tokens)))
        conn.commit(); conn.close()
        self.root.after(0, self.refresh_table)
        self.log("Аналіз завершено.")

    def _translate_batch(self):
        conn = sqlite3.connect(DB_PATH)
        rows = conn.execute("SELECT id, en_term FROM terms WHERE uk_term = '' OR uk_term IS NULL").fetchall()
        for row in rows:
            inputs = self.tokenizer(row[1], return_tensors="pt", padding=True)
            translated = self.model.generate(**inputs)
            uk_text = self.tokenizer.decode(translated[0], skip_special_tokens=True)
            conn.execute("UPDATE terms SET uk_term = ? WHERE id = ?", (uk_text, row[0]))
            conn.commit()
            self.log(f"Перекладено: {row[1]} -> {uk_text}")
        conn.close()
        self.root.after(0, self.refresh_table)

    def process_batch(self):
        threading.Thread(target=self._translate_batch, daemon=True).start()

    def refresh_table(self):
        for i in self.tree.get_children(): self.tree.delete(i)
        for row in sqlite3.connect(DB_PATH).execute("SELECT id, en_term, uk_term, length FROM terms").fetchall():
            self.tree.insert("", tk.END, values=row)

    def export_to_word(self):
        file_path = filedialog.asksaveasfilename(defaultextension=".docx")
        if not file_path: return
        doc = Document()
        doc.add_heading('Звіт зі словника термінів', 0)
        for length in [2, 3, 4]:
            doc.add_heading(f'{length}-компонентні терміни', level=1)
            table = doc.add_table(rows=1, cols=3)
            table.style = 'Table Grid'
            hdr = table.rows[0].cells
            hdr[0].text = "№"; hdr[1].text = "Термін (EN)"; hdr[2].text = "Переклад (UK)"
            rows = sqlite3.connect(DB_PATH).execute("SELECT en_term, uk_term FROM terms WHERE length = ?", (length,)).fetchall()
            for i, row in enumerate(rows, 1):
                cells = table.add_row().cells
                cells[0].text = str(i); cells[1].text = row[0]; cells[2].text = row[1]
        doc.save(file_path)
        messagebox.showinfo("Успіх", "Звіт створено!")

if __name__ == "__main__":
    root = tk.Tk()
    app = TermManagerApp(root)
    root.mainloop()